<a href="https://colab.research.google.com/github/simsekergun/DeviceMetrology/blob/main/Sidewalled_Dimension_Prediction_from_Dint.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

In [2]:
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Add, Dense, LeakyReLU, Input, Multiply, Activation, Dropout, LayerNormalization
from tensorflow.keras import regularizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

In [3]:
XTrain = pd.read_csv('https://raw.githubusercontent.com/simsekergun/DeviceMetrology/refs/heads/main/trapezoidal/mus_common_Dints_sim.csv')
XTrain.head(3)

,-60,-59,-58,-57,-56,-55,-54,-53,-52,-51,...,35,36,37,38,39,40,41,42,43,44
0,4.013100e+10,3.943500e+10,3.868900e+10,3.790700e+10,3.708600e+10,3.623400e+10,3.535300e+10,3.444500e+10,3.351900e+10,3.256800e+10,...,1.198100e+10,1.252200e+10,1.306600e+10,1.361600e+10,1.416000e+10,1.470000e+10,1.524300e+10,1.578400e+10,1.631500e+10,1.684100e+10
1,4.357400e+10,4.273900e+10,4.185800e+10,4.094600e+10,3.999600e+10,3.902200e+10,3.802000e+10,3.699700e+10,3.595600e+10,3.489700e+10,...,1.263400e+10,1.320900e+10,1.378700e+10,1.437300e+10,1.495300e+10,1.552900e+10,1.610900e+10,1.668800e+10,1.725700e+10,1.782200e+10
2,4.692300e+10,4.595200e+10,4.494000e+10,4.390100e+10,4.282700e+10,4.173300e+10,4.061300e+10,3.947800e+10,3.832500e+10,3.716100e+10,...,1.327000e+10,1.387800e+10,1.448900e+10,1.510800e+10,1.572500e+10,1.633600e+10,1.695100e+10,1.756800e+10,1.817500e+10,1.877700e+10


In [4]:
YTrain = pd.read_csv('https://raw.githubusercontent.com/simsekergun/DeviceMetrology/refs/heads/main/trapezoidal/wb_wt_h_sim.csv')
YTrain.head(3)

,wb,wt,h
0,840,820,670
1,840,820,675
2,840,820,680


In [5]:
XTest = pd.read_csv('https://raw.githubusercontent.com/simsekergun/DeviceMetrology/refs/heads/main/trapezoidal/mus_Dint_exp.csv')
XTest.head(1)

,-60,-59,-58,-57,-56,-55,-54,-53,-52,-51,...,35,36,37,38,39,40,41,42,43,44
0,4.601900e+10,4.500700e+10,4.426100e+10,4.222300e+10,4.093800e+10,4.084600e+10,3.844800e+10,3.841400e+10,3.707600e+10,3.470300e+10,...,1.191400e+10,1.248100e+10,1.289800e+10,1.332200e+10,1.375500e+10,1.418100e+10,1.436800e+10,1.553200e+10,1.580400e+10,1.606700e+10


In [6]:
xnormalizer = 7e10

XTrain_scaled = XTrain.values/xnormalizer
XTest_scaled = XTest.values/xnormalizer

In [7]:
column_means = YTrain.mean()
column_maxs = YTrain.max()
column_mins = YTrain.min()
Y_train_sc = (YTrain-column_mins)/(column_maxs-column_mins)
Y_train_scaled = Y_train_sc.values

## Prediction with a Sequential Neural Network

In [8]:
model2 = Sequential([
    Dense(256, input_shape=(54,), activation='tanh'),
    Dense(256, activation='tanh'),
    Dense(128, activation='tanh'),
    Dense(64, activation='tanh'),
    Dense(32, activation='tanh'),
    Dense(3, activation='tanh')  # Output layer with 3 units for your 3 outputs
])

# Compile the model
model2.compile(optimizer=Adam(learning_rate=0.001),
              loss='mse',
              metrics=['mae'])

# Train the model
history = model2.fit(XTrain_scaled, Y_train_scaled,
                    epochs=100,
                    batch_size=32,
                    verbose=1)

y_pred_scaled2 = model2.predict(XTest_scaled)

pred1 = y_pred_scaled2[0,0]*(column_maxs.iloc[0]-column_mins.iloc[0])+column_mins.iloc[0]
pred2 = y_pred_scaled2[0,1]*(column_maxs.iloc[1]-column_mins.iloc[1])+column_mins.iloc[1]
pred3 = y_pred_scaled2[0,2]*(column_maxs.iloc[2]-column_mins.iloc[2])+column_mins.iloc[2]
print(pred1, pred2, pred3)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.5842 - mae: 0.6201
Epoch 2/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.2836 - mae: 0.4354 
Epoch 3/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.2416 - mae: 0.4066 
Epoch 4/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.2106 - mae: 0.3874 
Epoch 5/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.1738 - mae: 0.3453 
Epoch 6/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.1493 - mae: 0.3321 
Epoch 7/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.1479 - mae: 0.3312 
Epoch 8/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.1326 - mae: 0.3100 
Epoch 9/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.1366 - mae: 0.3187 
Epoch 10/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.1308 - mae: 0.3059 
Epoch 11/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.1301 - mae: 0.3068 
Epoch 12/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.1311 - mae: 0.3068 
Epoch 13/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step 

## Prediction with SVM

In [14]:
from sklearn.svm import SVR
from sklearn.multioutput import MultiOutputRegressor

In [16]:
from sklearn.model_selection import GridSearchCV
svm_model = MultiOutputRegressor(
    SVR(kernel='rbf', C=100, epsilon=0.01)
)

param_grid = {
    'estimator__C': [1, 10, 100],
    'estimator__epsilon': [0.001, 0.01, 0.1]
}

grid = GridSearchCV(svm_model, param_grid, cv=3)
grid.fit(XTrain_scaled, Y_train_scaled)

svm_model = grid.best_estimator_

y_pred_scaled_svm = svm_model.predict(XTest_scaled)

pred1 = y_pred_scaled_svm[0,0]*(column_maxs.iloc[0]-column_mins.iloc[0])+column_mins.iloc[0]
pred2 = y_pred_scaled_svm[0,1]*(column_maxs.iloc[1]-column_mins.iloc[1])+column_mins.iloc[1]
pred3 = y_pred_scaled_svm[0,2]*(column_maxs.iloc[2]-column_mins.iloc[2])+column_mins.iloc[2]

print(pred1, pred2, pred3)

862.2439239887924 831.9201773069833 671.925135355545


## Prediction with XGB

In [17]:
from xgboost import XGBRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import RandomizedSearchCV

xgb_base = XGBRegressor(
    objective='reg:squarederror',
    n_estimators=200,
    verbosity=0
)

xgb_model = MultiOutputRegressor(xgb_base)

param_dist = {
    'estimator__n_estimators': [100, 200, 300, 500],
    'estimator__max_depth': [3, 5, 7, 10],
    'estimator__learning_rate': [0.01, 0.05, 0.1, 0.2],
    'estimator__subsample': [0.7, 0.8, 1.0],
    'estimator__colsample_bytree': [0.7, 0.8, 1.0],
    'estimator__gamma': [0, 0.1, 0.2],
    'estimator__reg_alpha': [0, 0.01, 0.1],
    'estimator__reg_lambda': [1, 5, 10]
}

random_search = RandomizedSearchCV(
    xgb_model,
    param_distributions=param_dist,
    n_iter=30,          # increase if you want better tuning
    cv=3,
    verbose=1,
    n_jobs=-1
)

random_search.fit(XTrain_scaled, Y_train_scaled)

best_xgb = random_search.best_estimator_
print("Best params:", random_search.best_params_)

Fitting 3 folds for each of 30 candidates, totalling 90 fits
Best params: {'estimator__subsample': 1.0, 'estimator__reg_lambda': 5, 'estimator__reg_alpha': 0.1, 'estimator__n_estimators': 300, 'estimator__max_depth': 3, 'estimator__learning_rate': 0.1, 'estimator__gamma': 0, 'estimator__colsample_bytree': 0.8}


In [18]:
y_pred_scaled_xgb = best_xgb.predict(XTest_scaled)

pred1 = y_pred_scaled_xgb[0,0]*(column_maxs.iloc[0]-column_mins.iloc[0])+column_mins.iloc[0]
pred2 = y_pred_scaled_xgb[0,1]*(column_maxs.iloc[1]-column_mins.iloc[1])+column_mins.iloc[1]
pred3 = y_pred_scaled_xgb[0,2]*(column_maxs.iloc[2]-column_mins.iloc[2])+column_mins.iloc[2]

print(pred1, pred2, pred3)

860.4151940345764 850.7586240768433 670.2079105516896


## Decision Tree

In [19]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import RandomizedSearchCV

dt = MultiOutputRegressor(DecisionTreeRegressor())

param_dist_dt = {
    'estimator__max_depth': [None, 5, 10, 20, 30],
    'estimator__min_samples_split': [2, 5, 10],
    'estimator__min_samples_leaf': [1, 2, 4],
    'estimator__max_features': [None, 'sqrt', 'log2']
}

dt_search = RandomizedSearchCV(
    dt,
    param_distributions=param_dist_dt,
    n_iter=20,
    cv=3,
    verbose=1,
    n_jobs=-1
)

dt_search.fit(XTrain_scaled, Y_train_scaled)

best_dt = dt_search.best_estimator_
print("Best DT params:", dt_search.best_params_)

Fitting 3 folds for each of 20 candidates, totalling 60 fits
Best DT params: {'estimator__min_samples_split': 10, 'estimator__min_samples_leaf': 4, 'estimator__max_features': 'log2', 'estimator__max_depth': 20}


In [20]:
y_pred_scaled_dt = best_dt.predict(XTest_scaled)

pred1 = y_pred_scaled_dt[0,0]*(column_maxs.iloc[0]-column_mins.iloc[0])+column_mins.iloc[0]
pred2 = y_pred_scaled_dt[0,1]*(column_maxs.iloc[1]-column_mins.iloc[1])+column_mins.iloc[1]
pred3 = y_pred_scaled_dt[0,2]*(column_maxs.iloc[2]-column_mins.iloc[2])+column_mins.iloc[2]

print(pred1, pred2, pred3)

868.0 837.5 685.0


## Random Forest

In [21]:
from sklearn.ensemble import RandomForestRegressor

rf = MultiOutputRegressor(RandomForestRegressor())

param_dist_rf = {
    'estimator__n_estimators': [100, 200, 300, 500],
    'estimator__max_depth': [None, 10, 20, 30],
    'estimator__min_samples_split': [2, 5, 10],
    'estimator__min_samples_leaf': [1, 2, 4],
    'estimator__max_features': ['sqrt', 'log2']
}

rf_search = RandomizedSearchCV(
    rf,
    param_distributions=param_dist_rf,
    n_iter=20,
    cv=3,
    verbose=1,
    n_jobs=-1
)

rf_search.fit(XTrain_scaled, Y_train_scaled)

best_rf = rf_search.best_estimator_
print("Best RF params:", rf_search.best_params_)

Fitting 3 folds for each of 20 candidates, totalling 60 fits
Best RF params: {'estimator__n_estimators': 100, 'estimator__min_samples_split': 5, 'estimator__min_samples_leaf': 1, 'estimator__max_features': 'log2', 'estimator__max_depth': 30}


In [22]:
y_pred_scaled_rf = best_rf.predict(XTest_scaled)

In [23]:
pred1 = y_pred_scaled_rf[0,0]*(column_maxs.iloc[0]-column_mins.iloc[0])+column_mins.iloc[0]
pred2 = y_pred_scaled_rf[0,1]*(column_maxs.iloc[1]-column_mins.iloc[1])+column_mins.iloc[1]
pred3 = y_pred_scaled_rf[0,2]*(column_maxs.iloc[2]-column_mins.iloc[2])+column_mins.iloc[2]

print(pred1, pred2, pred3)

863.6886904761905 832.8663095238095 684.05
